# Lesson 200.24 — Governor Integration with Lean 4: Multi-Simplex Regge Triangulations and Hinge Holonomy Verification

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Lean 4 kernel specifications (`Level21DiscreteRegge.lean`) and target checkers (`stage_021.py`).
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.

## 1. The Big Idea: Connecting Runtime Governors to Formal Proof Kernels

**Core Concept:** 4D Regge multi-simplex Gram matrix positivity and hinge holonomy verification under Stage 21 contract clauses

### The Four-Tier Perspective
- **Tangible (Age 5–10):** Checking that building blocks fit both on edges and in the interior.
- **Algorithmic (Age 11–17):** Deterministic verification of Gram determinants and loop words.
- **Discrete Math (Undergrad):** Exact rational minors ($M_1, M_2, M_3, M_4 > 0$) and $SO(4)$ Lie group holonomy.
- **Foundational (Age 50–500):** Mechanized proof kernel invariants bounding discrete spacetime geometries.


In [ ]:
from fractions import Fraction as Q
from itertools import combinations


def det_q(matrix):
    rows = [[Q(x) for x in row] for row in matrix]
    n = len(rows)
    sign = Q(1)
    for col in range(n):
        pivot = next((r for r in range(col, n) if rows[r][col] != 0), None)
        if pivot is None:
            return Q(0)
        if pivot != col:
            rows[col], rows[pivot] = rows[pivot], rows[col]
            sign = -sign
        scale = rows[col][col]
        sign *= scale
        for r in range(col + 1, n):
            factor = rows[r][col] / scale
            for c in range(col, n):
                rows[r][c] -= factor * rows[col][c]
    return sign


# 1. Define regular unit 4-simplex with 10 edges of length 1 (all squared lengths = 1)
edges = list(combinations(range(5), 2))
lengths = {e: Q(1) for e in edges}

# 2. Build 4x4 Gram matrix based at vertex 0
rest = [1, 2, 3, 4]


def sq(a, b):
    return Q(0) if a == b else lengths[(min(a, b), max(a, b))]


gram = [[(sq(0, a) + sq(0, b) - sq(a, b)) / Q(2) for b in rest] for a in rest]

# 3. Compute leading principal minors
minors = [det_q([row[:k] for row in gram[:k]]) for k in range(1, 5)]
expected_minors = [Q(1), Q(3, 4), Q(1, 2), Q(5, 16)]
assert minors == expected_minors
assert all(m > 0 for m in minors)
print("Regular unit 4-simplex Gram matrix strictly positive definite:", minors)

## 2. Scope & Claim Boundary

| CAN Show in this Lesson | CANNOT Show in this Lesson |
| :--- | :--- |
| Exact discrete verification of 4-simplex Gram positivity and hinge holonomy over rational arithmetic. | Generalization to unconstrained smooth Lorentzian manifolds without discrete triangulation bounds. |
| Enforcement of Stage 21 governor contract clauses C1-C4 with typed refusal receipts. | Universal proof outside declared simplicial complexes. |


## 3. Diagnostic Probe (2026 Micro-Attention Check)

**Question:** Under the Stage 21 governor contract (`stage_021.py`), what typed refusal code is emitted when a candidate declares `closure: "identity"` on a loop whose scalar exponent sum is zero, but whose ordered $SO(4)$ matrix product is not the identity?

- **A:** `R021_GRAM_NOT_POSITIVE`
- **B:** `R021_SCALAR_SUM_CLOSURE`
- **C:** `R021_NOT_ROTATION`


In [ ]:
# Interactive Diagnostic Probe Check
student_choice = "B"
assert student_choice == "B", (
    f"Incorrect choice: {student_choice}. Expected B (R021_SCALAR_SUM_CLOSURE)."
)


def mat_mul4(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(4)) for j in range(4)] for i in range(4)]


def transpose4(A):
    return [[A[j][i] for j in range(4)] for i in range(4)]


def identity4():
    return [[Q(1 if i == j else 0) for j in range(4)] for i in range(4)]


def evaluate_stage_021(payload):
    lengths = {}
    for u, v, val in payload["squared_lengths"]:
        lengths[(min(u, v), max(u, v))] = Q(val)
    if set(lengths.keys()) != set(combinations(range(5), 2)):
        return {"verdict": "R_SCHEMA", "reason": "Expected 10 edge lengths on 5 vertices"}

    rest = [1, 2, 3, 4]

    def sq(a, b):
        return Q(0) if a == b else lengths[(min(a, b), max(a, b))]

    gram = [[(sq(0, a) + sq(0, b) - sq(a, b)) / Q(2) for b in rest] for a in rest]

    for k in range(1, 5):
        submatrix = [row[:k] for row in gram[:k]]
        if det_q(submatrix) <= Q(0):
            return {"verdict": "R021_GRAM_NOT_POSITIVE", "failed_minor": k}

    I4 = identity4()
    generators = {}
    for name, mat in payload.get("generators", []):
        mat_q = [[Q(x) for x in row] for row in mat]
        if mat_mul4(mat_q, transpose4(mat_q)) != I4 or det_q(mat_q) != Q(1):
            return {"verdict": "R021_NOT_ROTATION", "generator": name}
        generators[name] = mat_q

    for loop in payload.get("loops", []):
        hinge = loop["hinge"]
        if len(set(hinge)) != 3 or not set(hinge) <= set(range(5)):
            return {"verdict": "R_SCHEMA", "reason": "Invalid hinge"}

        product = identity4()
        exponents = {name: 0 for name in generators}
        for letter in loop["word"]:
            inv = letter.endswith("-")
            name = letter[:-1] if inv else letter
            factor = transpose4(generators[name]) if inv else generators[name]
            product = mat_mul4(product, factor)
            exponents[name] += -1 if inv else 1

        closed = product == I4
        if loop["closure"] == "identity" and not closed:
            if all(exp == 0 for exp in exponents.values()):
                return {"verdict": "R021_SCALAR_SUM_CLOSURE", "hinge": hinge}
            return {"verdict": "R021_CLOSURE_MISMATCH", "hinge": hinge}

    return {
        "verdict": "ADMITTED",
        "gram_minors": [det_q([r[:k] for r in gram[:k]]) for k in range(1, 5)],
    }


# Test probe payload: Lean 4 countermodel loop
payload_negative = {
    "squared_lengths": [[a, b, "1/1"] for a, b in combinations(range(5), 2)],
    "generators": [
        [
            "Rx",
            [
                ["1", "0", "0", "0"],
                ["0", "0", "-1", "0"],
                ["0", "1", "0", "0"],
                ["0", "0", "0", "1"],
            ],
        ],
        [
            "Ry",
            [
                ["0", "0", "1", "0"],
                ["0", "1", "0", "0"],
                ["-1", "0", "0", "0"],
                ["0", "0", "0", "1"],
            ],
        ],
    ],
    "loops": [{"closure": "identity", "hinge": [0, 1, 2], "word": ["Rx", "Ry", "Rx-", "Ry-"]}],
}
res_neg = evaluate_stage_021(payload_negative)
assert res_neg["verdict"] == "R021_SCALAR_SUM_CLOSURE"
print("Stage 21 governor correctly emitted R021_SCALAR_SUM_CLOSURE. Probe PASSED.")

## 4. Student Exercise

**Task:** Assert that a degenerate 4-simplex with a collapsed edge is refused with `R021_GRAM_NOT_POSITIVE`, and assert that a genuinely closed loop is admitted.

### Hidden Solution
<details>
<summary>Click to view solution</summary>

```python
# Exercise solution
payload_deg = dict(payload_negative)
payload_deg['squared_lengths'] = [[a, b, '0/1' if (a, b) == (0, 1) else '1/1'] for a, b in combinations(range(5), 2)]
res_deg = evaluate_stage_021(payload_deg)
assert res_deg['verdict'] == 'R021_GRAM_NOT_POSITIVE'

payload_pos = dict(payload_negative)
payload_pos['loops'] = [{'closure': 'identity', 'hinge': [0, 1, 2], 'word': ['Rx', 'Rx-']}]
res_pos = evaluate_stage_021(payload_pos)
assert res_pos['verdict'] == 'ADMITTED'
assert res_pos['gram_minors'] == [Q(1), Q(3, 4), Q(1, 2), Q(5, 16)]
print('Exercise verified successfully!')
```
</details>

## 5. Anchors

| Anchor Type | Target Link |
| :--- | :--- |
| **Curriculum Anchor** | `stage_021.py` |
| **Formal Lean Kernel** | `Level21DiscreteRegge.lean` |
| **Authority Effect** | `None` |
| **Zero-Float Law** | Enforced (`fractions.Fraction` or exact integers) |


In [ ]:
# Exercise implementation
payload_deg = dict(payload_negative)
payload_deg["squared_lengths"] = [
    [a, b, "0/1" if (a, b) == (0, 1) else "1/1"] for a, b in combinations(range(5), 2)
]
res_deg = evaluate_stage_021(payload_deg)
assert res_deg["verdict"] == "R021_GRAM_NOT_POSITIVE"

payload_pos = dict(payload_negative)
payload_pos["loops"] = [{"closure": "identity", "hinge": [0, 1, 2], "word": ["Rx", "Rx-"]}]
res_pos = evaluate_stage_021(payload_pos)
assert res_pos["verdict"] == "ADMITTED"
assert res_pos["gram_minors"] == [Q(1), Q(3, 4), Q(1, 2), Q(5, 16)]
print("Governor integration with Lean 4 Level 21 Regge fully verified!")